# 06. Operational Space and Data-Driven Plasma Analysis

Sessions 01–05 went deeper into one discharge at a time. This session turns the other way: many
discharges, many equilibrium slices and many events become a **population of plasma states**.

```text
Session 01   load and understand a discharge
Session 02   operate and form the plasma
Session 03   reconstruct the plasma state
Session 04   measure perturbations and events
Session 05   model stability and 3-D response
      ↓
Session 06   compare many plasma states, and learn from the database      <- here
```

$$
\text{discharges} \rightarrow \text{representative features} \rightarrow \text{population}
\rightarrow \text{operational space} \rightarrow \text{limits / regimes / performance}
\rightarrow \text{data-driven representations}
$$

The design of the session follows issue [#1091](https://github.com/VEST-Tokamak/vaft/issues/1091).

## Session Overview

**The question.** What can be extracted from a tokamak database once individual discharges are
represented consistently — and what can the resulting population tell us, and not tell us, about
operating regimes, limits, confinement and rare events?

By the end of this session you will be able to:

- tell shot-level, time-slice-level and event-level records apart, and say which one a question needs;
- treat the choice of a representative state as a modelling decision, stated and not hidden;
- check coverage, missingness and validity **before** drawing a physics conclusion;
- read distributions and parameter-space occupancy, and tell an **observed envelope** from an
  **accessible space** and from a **physical limit**;
- overlay reference limits only after asking where they come from and whether they apply;
- move from engineering variables to dimensionless ones, and from one $\tau_E$ to a population of them;
- recognise confounding and pseudo-replication in a pooled regression;
- place statistics, machine learning and representation learning on one ladder — and say which rungs
  VAFT has today.

**The population.** Offline, the six packaged VEST samples: 39915, 40600, 41524, 41672, 45531 and
48224. Six discharges are not a database, and the session says so wherever it matters; what they *do*
give is every step of the workflow on real VEST records, including the ones that fail. In lab mode
the same cells run on a shot range from the VEST database through `vaft.database.summary`.

**The rule.** Every table comes from VAFT's canonical summary presets — the same extraction the
database uses — never from tutorial-only logic.

## Physical Context

### 1. From one discharge to a database

For a state $i$, write the quantities of interest as a vector,
$\mathbf x_i = [I_p, B_T, q_{95}, \beta, l_i, \kappa, \delta, \bar n_e, \tau_{\mathrm{pulse}}, \dots]$,
and the database as the matrix whose rows are the $\mathbf x_i$. A database-scale study begins only
when **the same quantity is extracted with the same definition and provenance** in every row.

### 2. The unit of analysis

| unit | example | typical question |
| --- | --- | --- |
| shot | max $I_p$, pulse duration, shot class | what kind of discharge was produced? |
| time slice | $q_{95}$, $\beta_N$, $l_i$, $\kappa$ | where did the plasma state exist? |
| event | mode onset, current quench, termination | from what state did a transition occur? |

A representative value answers a question: maximum $I_p$, flat-top mean, maximum $\beta_N$, a fixed
time, the equilibrium nearest a mode onset, the state before an event. The chain is

$$
\boxed{\text{scientific question} \rightarrow \text{sampling rule} \rightarrow \text{representative value}}
$$

and never "one row per shot" without a stated rule.

### 3. Observed envelope, accessible space, physical limit

A database can be empty in a region because the condition was never attempted, the actuators could
not reach it, operators avoided it, diagnostics or reconstructions did not cover it, a quality filter
removed it — or because the plasma was unstable there. **An empty region is not by itself an
operating limit.** Physical interpretation needs more: machine-accessibility constraints, event labels
(session 04), stability calculations (session 05), sourced reference limits, performance.

### 4. Reference limits, with their provenance

- **Low $q$**: $q_a$ or $q_{95}$ near 2 and the current-profile peaking $l_i$ — the classic
  $(q_a, l_i)$ disruption diagram.
- **Beta**: $\beta_N = \beta[\%]\,aB_T/I_p[\mathrm{MA}]$ and Troyon-type limits $\beta_N \lesssim 3$.
- **Density**: the Greenwald density $n_G = I_p[\mathrm{MA}]/(\pi a^2)\times 10^{20}\,\mathrm{m^{-3}}$
  and $f_G = \bar n_e/n_G$.

A formula existing in a code base is not a reason to believe it. Each reference must be checked for
source, units and domain of validity before it is drawn over data.

### 5. Engineering and dimensionless variables

Engineering variables — $I_p, B_T, \bar n_e, T, P_{\mathrm{loss}}, R, a, \kappa$ — say how a
discharge was run. The dimensionless ones — $\rho_* = \rho_i/a$, $\beta$, $\nu_*$, $q$,
$\epsilon = a/R$ — say what physics regime it was in. Two discharges with different engineering
parameters can be physically similar; that is the bridge to comparisons between machines.

### 6. Confinement as a population quantity

$\tau_E = W_{\mathrm{th}}/P_{\mathrm{loss}}$ for one discharge becomes $\{\tau_{E,i}\}$ for a
database. A power law
$\tau_E = C\,I_p^{\alpha_I} B_T^{\alpha_B} P^{\alpha_P} \bar n_e^{\alpha_n} R^{\alpha_R}\cdots$
is a log-linear regression, and its comparison with a reference scaling is the H factor
$H = \tau_E^{\mathrm{measured}}/\tau_E^{\mathrm{scaling}}$. Similarity assumptions constrain which
exponent combinations make sense (Kadomtsev).

### 7. Correlation, confounding, selection

$\mathrm{corr}(I_p, q_{95})$ is not a causal statement. Derived quantities share variables,
actuators are coupled, campaigns change several things at once, diagnostics select which shots are
analysable, and missing data are rarely missing at random. Slices of one discharge are not
independent samples of the machine.

## Load / Prepare Data

### The population, offline

Each sample is loaded and passed through the canonical summary presets. `get_summary_preset(name)`
is the preset the database summary uses; its extractor turns one ODS into the preset's rows. A
preset that cannot run on a shot is recorded with its reason — that record *is* data.

In [ ]:
import logging
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import vaft
from vaft.formula import equilibrium as eq_formula
from vaft.formula import stability
from vaft.process.statistical_analysis import perform_ols_regression
from vaft.process.transients import current_quench

# The extractors narrate every repair they make; the coverage table below reports what matters.
# Lab mode turns the database logger back on (Part II), where a skipped shot must be visible.
logging.getLogger("vaft").setLevel(logging.ERROR)

In [ ]:
SHOTS = (39915, 40600, 41524, 41672, 45531, 48224)
PRESETS = ("shot_overview", "equilibrium_global", "core_profiles", "volume_averaged")

samples = {shot: vaft.omas.sample_ods(shot) for shot in SHOTS}
rows = {preset: [] for preset in PRESETS}
failures = []
for shot, ods in samples.items():
    for preset in PRESETS:
        try:
            rows[preset].extend(vaft.database.get_summary_preset(preset).extractor(ods, shot))
        except Exception as error:   # a preset that cannot run on this shot is itself a record
            failures.append({"shot": shot, "preset": preset, "reason": f"{type(error).__name__}: {error}"[:90]})
tables = {preset: pd.DataFrame(found) for preset, found in rows.items()}

# Lab mode: the same presets over a database shot range (needs HSDS credentials).
# tables = {preset: vaft.database.summary(shot_range=(39000, 49000), preset=preset) for preset in PRESETS}

for preset, table in tables.items():
    print(f"{preset:18s} {len(table):3d} rows from {table['shot'].nunique() if len(table) else 0} shots")
for failure in failures:
    print(f"  {failure['shot']} {failure['preset']}: {failure['reason']}")

Four presets, six shots, and already three different answers to "how big is the population": five
shot-level rows, 35 equilibrium slices, and one kinetic state. One shot has no
plasma-current record in its sample and so no shot overview; two have no equilibrium at all.

## Guided Analysis

### Step 1 — The unit of analysis, and who dominates it

In [ ]:
shots_table = tables["shot_overview"]
slices = tables["equilibrium_global"]
per_shot = slices.groupby("shot").size()

print(f"shot-level records : {len(shots_table)}")
print(f"slice-level records: {len(slices)} from {slices['shot'].nunique()} shots")
for shot, count in per_shot.items():
    print(f"  {shot}: {count:2d} slices ({count / len(slices):.0%} of the slice population)")

Pool the slices and one discharge supplies more than half of the population. A histogram of $q_{95}$
over "all slices" is mostly a histogram of that one discharge's evolution. Slices of one shot are
correlated — they share the machine settings, the wall, the fuelling, the reconstruction — so they
are not independent samples of the machine. This is **pseudo-replication**, and it is the first
thing to check before any statistic.

In [ ]:
figure, axes = plt.subplots(figsize=(7, 4.2))
axes.plot(shots_table["pulse_duration_s"] * 1e3, shots_table["max_ip_kA"], "o")
for _, shot_row in shots_table.iterrows():
    axes.annotate(f"#{int(shot_row['shot'])}", (shot_row["pulse_duration_s"] * 1e3, shot_row["max_ip_kA"]),
                  textcoords="offset points", xytext=(5, 4), fontsize=8)
axes.set_xlabel("plasma pulse duration [ms]")
axes.set_ylabel(r"max $I_p$ [kA]")
axes.set_title("Shot level: one point per discharge")
axes.grid(alpha=0.4)
plt.show()

### Step 2 — Coverage and validity before physics

The funnel from "all shots" to "shots a given analysis can use" narrows at every step. The coverage
fraction of a quantity is $C_X = N(X\ \text{available})/N(\text{relevant records})$.

In [ ]:
coverage = pd.DataFrame(
    {preset: [int((table["shot"] == shot).sum()) if len(table) else 0 for shot in SHOTS]
     for preset, table in tables.items()},
    index=pd.Index(SHOTS, name="shot"),
)
print(coverage.to_string())
print()
for preset in PRESETS:
    covered = int((coverage[preset] > 0).sum())
    print(f"C({preset:18s}) = {covered}/{len(SHOTS)} = {covered / len(SHOTS):.2f}")

valid = slices["ip_kA"].abs() > 0
print(f"\nrule 1 -- zero plasma current (failed or post-discharge reconstruction): {int((~valid).sum())} removed")
slices = slices[valid].copy()

# Rule 2: states of the discharge before it ends. Each shot's own current quench, measured on its
# plasma-current record as in session 04, sets the cut; reconstructions during the quench are not
# states a limit study is about, and their profile quantities are the least trustworthy.
quench_start = {}
for shot, ods in samples.items():
    if "magnetics.ip.0.data" in ods:
        quench_start[shot] = current_quench(np.asarray(ods["magnetics.ip.0.time"], dtype=float),
                                            np.asarray(ods["magnetics.ip.0.data"], dtype=float)).time_80
cut = slices["shot"].map(quench_start)
before = cut.isna() | (slices["time_s"] < cut)
dropped = slices[~before]
print(f"rule 2 -- at or after the shot's own current-quench 80 % time: {len(dropped)} removed "
      f"(their l_i: {dropped['li_3'].min():.2f}-{dropped['li_3'].max():.2f})")
print(f"          shots with no current record keep their slices unjudged: "
      f"{sorted(int(s) for s in slices.loc[cut.isna(), 'shot'].unique())}")
slices = slices[before].copy()
print(f"slices kept: {len(slices)}")

Missing data are part of the experiment. Here the kinetic presets cover one shot of six, the
equilibrium preset four, and two slices carry no plasma current at all — reconstructions after the
current had gone. The second rule removes every reconstruction from the moment the shot's own
current quench begins (its 80 % time, measured as in session 04): their $l_i$ runs to values no
equilibrium of this machine has — a reconstruction of a quench, not a state of the discharge. A
cruder "half of the peak current" cut would keep ten of those slices. Where a shot has no current
record, the rule cannot judge its slices, and the output says so. Both are stated rules, not silent
cleanups, and the counts of what they removed belong in any result.

### Step 3 — A representative state is a modelling decision

Reduce each shot to one equilibrium slice, by two different rules, and compare what comes out.

In [ ]:
# Two sampling rules. #1233 will give these a public, named API; until then the rule is stated here.
peak_rows = slices["ip_kA"].abs().groupby(slices["shot"]).idxmax()   # |Ip|: the sign is a convention
at_max_ip = slices.loc[peak_rows].set_index("shot")
at_max_w = slices.loc[slices.groupby("shot")["energy_mhd_J"].idxmax()].set_index("shot")

print(f"{'shot':>6} {'rule':>12} {'t [ms]':>7} {'Ip [kA]':>8} {'q95':>6} {'li':>6} {'kappa':>6}")
for shot in at_max_ip.index:
    for label, table in (("max Ip", at_max_ip), ("max W_mhd", at_max_w)):
        row = table.loc[shot]
        print(f"{shot:>6} {label:>12} {row['time_s'] * 1e3:7.1f} {row['ip_kA']:8.1f} "
              f"{row['q_95']:6.2f} {row['li_3']:6.3f} {row['elongation']:6.2f}")

On most shots the two rules pick the same slice; on 41672 they pick different times, and $q_{95}$
moves with them. Neither rule is wrong — they answer different questions ("what was the highest-current
state?", "what was the highest-energy state?"). A population built with one rule is not comparable
with one built with the other, and the rule belongs in the table's provenance.

### Step 4 — Distributions

An empirical cumulative distribution, $F_X(x) = P(X \le x)$, shows a whole sample without choosing a
bin width. Draw it twice: over all valid slices, and over one representative state per shot.

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for axis, column, label in zip(axes, ("q_95", "li_3", "elongation"), (r"$q_{95}$", r"$l_i$", r"$\kappa$")):
    for data, style, name in ((slices[column], "-", f"all slices (N={len(slices)})"),
                              (at_max_ip[column], "o--", f"max-Ip state per shot (N={len(at_max_ip)})")):
        values = np.sort(data.dropna().to_numpy())
        axis.step(values, np.arange(1, values.size + 1) / values.size, style, where="post", label=name)
    axis.set_xlabel(label)
    axis.set_ylabel("ECDF")
    axis.grid(alpha=0.4)
axes[0].legend(fontsize="small")
figure.suptitle("The same quantity, two units of analysis")
plt.show()
print(slices[["q_95", "li_3", "elongation"]].describe(percentiles=[0.1, 0.5, 0.9]).round(3).to_string())

### Step 5 — Operational space, and what an envelope is not

The $(q, l_i)$ plane is the classic current-limit diagram: low $q$ and peaked current profiles
(high $l_i$) are where current-driven disruptions live. `vaft.formula.stability.empirical_li_qa`
carries the points of the JET survey behind that diagram — digitised from a published figure, for a
different machine. It is a **reference drawn for comparison, not a VEST limit.**

In [ ]:
qa_survey, li_survey = stability.empirical_li_qa()

figure, axes = plt.subplots(figsize=(7.5, 5))
axes.plot(qa_survey, li_survey, "x", color="0.55", label="JET survey points (Wesson et al.), reference only")
for shot, group in slices.groupby("shot"):
    axes.plot(group["q_95"], group["li_3"], "o-", markersize=4, label=f"#{shot} slices")
axes.plot(at_max_ip["q_95"], at_max_ip["li_3"], "k*", markersize=13, label="max-Ip state per shot")
axes.axvline(2.0, color="r", linestyle="--", linewidth=1, label="q = 2")
axes.set_xlabel(r"$q_{95}$ (VEST) / $q_a$ (survey)")
axes.set_ylabel(r"$l_i$")
axes.set_title("Observed VEST states in the (q, li) plane: an observed envelope, not a limit")
axes.legend(fontsize="small", loc="upper right")
axes.grid(alpha=0.4)
plt.show()

Every VEST state sits far from $q = 2$, and each shot draws a track as its current evolves. The
empty lower-$q$ region is exactly the case of Section 3: the population says where VEST *was*, not
where it *could not go*. Whether low $q$ is reachable is a question for the PF and power-supply
limits, and whether it would be stable is a question for session 05's DCON and RDCON on states near
the edge of the cloud. Note also that the axes differ — $q_{95}$ for VEST, $q_a$ for the survey — so
even the reference is only indicative.

### Step 6 — Beta: a quantity whose validity comes first

$\beta_N$ is in every slice row. Before plotting it against a Troyon line, read its magnitude.

In [ ]:
print(slices.groupby("shot")[["beta_normal", "beta_pol", "energy_mhd_J"]].median().round(3).to_string())
print("\nconstrained by kinetic profiles:", sorted(int(s) for s in tables["core_profiles"]["shot"].unique()))

row = at_max_ip.iloc[0]
check = stability.beta_N_from_beta_a_B0_Ip(100 * row["beta_tor"], row["minor_radius_m"],
                                          row["vacuum_b0_T"], row["ip_kA"] / 1e3)
print(f"\nrecomputed from beta_t, a, B0, Ip for #{at_max_ip.index[0]}: {check:.3f} (stored {row['beta_normal']:.3f})")

The stored $\beta_N$ is consistent with its own inputs — and the rows are not all the same kind of
number. The three magnetics-only reconstructions have no pressure constraint: their $\beta_N$ is a
few hundredths to about a tenth, and a magnetics-only EFIT of these samples is known to carry far
less pressure than force balance needs (issue #386). The one reconstruction constrained by kinetic
profiles, 48224, has the highest $\beta_N$, and Step 8 finds its $\beta_t$ between the Thomson
electron-only and electron-plus-ion estimates. So a Troyon diagram of the pooled column would mix two
different quantities under one name: "far below the beta limit" would be true of the
reconstructions and say nothing about those plasmas. **Validity comes before physics**: a column's
constraint set is part of its definition, and pressure without a pressure constraint is not a
beta-limit variable.

The same audit applies to the formulas. `stability.kink_stability_criterion` compares $\beta_N$ with
$2.8\,q_{95}$; its own docstring records that the factor $q_{95}$ has no source (#350). Such a
function is not drawn here as a limit. The Troyon picture itself is:

In [ ]:
vaft.diagram.troyon()

### Step 7 — Density limit, where the population has a density

Only one sample carries kinetic profiles, so the Greenwald fraction is a single point — a coverage
fact, not a physics one.

In [ ]:
kinetic = tables["core_profiles"]
for _, state in kinetic.iterrows():
    shot = int(state["shot"])
    geometry = slices[(slices["shot"] == shot)].iloc[
        int(np.argmin(np.abs(slices.loc[slices["shot"] == shot, "time_s"].to_numpy() - state["time_s"])))]
    n_g = stability.greenwald_density(state["ip_kA"] / 1e3, geometry["minor_radius_m"])   # in 1e19 m^-3
    f_g = stability.greenwald_fraction(state["ne_line_1e19_m3"], n_g)                    # both in 1e19 m^-3
    print(f"#{shot} at {state['time_s'] * 1e3:.0f} ms: line-averaged n_e = {state['ne_line_1e19_m3']:.2f}e19 m^-3, "
          f"n_G = {n_g:.2f}e19 m^-3, f_G = {f_g:.3f}")

In [ ]:
vaft.diagram.hugill()

The kinetic state sits at about a tenth of the Greenwald density. (`greenwald_density` returns
$10^{19}\,\mathrm{m^{-3}}$, not the literature's $10^{20}$ — read a formula's units before its value.) One point cannot place VEST's
density envelope; it can show that the formula, its units and the pairing of kinetic and equilibrium
times (by time, not by index) are in place for when the database supplies more. Greenwald is itself
a family of limits — global, edge, sustainment, disruptive — whose differences are the subject of
#1068.

### Step 8 — From engineering to dimensionless variables

Transform the kinetic state's engineering variables into the dimensionless set.

In [ ]:
state = kinetic.iloc[0]
epsilon = state["inverse_aspect_ratio"]
n_e, t_e, b_t = state["ne_volume_1e19_m3"] * 1e19, state["te_mean_eV"], state["b_t_T"]
rho_star = eq_formula.rho_star_from_M_T_B_R_epsilon(1.0, t_e, b_t, state["major_radius_m"], epsilon)   # hydrogen
beta_t = eq_formula.beta_t_from_n_T_B(n_e, t_e, b_t)
nu_star = eq_formula.nu_star_from_n_T_B_R_epsilon_kappa_I(n_e, t_e, b_t, state["major_radius_m"], epsilon,
                                                        state["elongation"], state["ip_kA"] * 1e3)
q_cyl = eq_formula.q_cyl_from_B_R_epsilon_kappa_I(b_t, state["major_radius_m"], epsilon, state["elongation"],
                                                  state["ip_kA"] * 1e3)
print(f"engineering : Ip = {state['ip_kA']:.0f} kA, B_T = {b_t:.3f} T, n_e = {n_e:.2e} m^-3, "
      f"T_e = {t_e:.0f} eV, R = {state['major_radius_m']:.2f} m, epsilon = {epsilon:.2f}")
print(f"dimensionless: rho* = {rho_star:.3f}, beta_t = {beta_t:.3f} %, nu* = {nu_star:.2f}, "
      f"q_cyl = {q_cyl:.2f}, epsilon = {epsilon:.2f}")
same_shot = slices[slices["shot"] == int(state["shot"])]
nearest = same_shot.iloc[int(np.argmin(np.abs(same_shot["time_s"].to_numpy() - state["time_s"])))]
print(f"\nbeta_t, kinetic with T_i = T_e (n T summed over electrons and ions): {beta_t:.2f} %; "
      f"electrons alone: {beta_t / 2:.2f} %")
print(f"beta_t of the reconstruction {abs(nearest['time_s'] - state['time_s']) * 1e3:.1f} ms from the "
      f"kinetic time: {nearest['beta_tor'] * 100:.2f} %")
print(f"q95 of that reconstruction: {nearest['q_95']:.2f}; q_cyl: {q_cyl:.2f}")

These are the Verdoolaege engineering forms, with the hydrogen mass of VEST's plasma (session 03),
the electron temperature standing in for the ion one, and volume averages multiplied rather than
$\langle nT\rangle$ — each an assumption to state beside the numbers. The kinetic $\beta_t$ is
therefore an electron measurement doubled by assuming $T_i = T_e$; with electrons alone it falls
below the reconstruction's, so the two bracket each other rather than agree.

$q_{\mathrm{cyl}}$ near 2 beside a $q_{95}$ near 6 is not a contradiction: the cylindrical safety
factor ignores the toroidal enhancement that dominates at $\epsilon \approx 0.7$, and the
formula wants the *area* elongation where the summary supplies the boundary one. The
low-aspect-ratio plasma is exactly where $q$ definitions part company. VAFT also
has other collisionality definitions (in `vaft.formula.neoclassical` and `vaft.formula.stability`);
they are not interchangeable, and a population study must pick one and say which. $\rho_*$ of a few
percent and $\epsilon$ near 0.7 are what make VEST a spherical tokamak in dimensionless terms, far
from a conventional-aspect-ratio device with the same current.

### Step 9 — Confinement and H factors

The `core_profiles` preset already carries $\tau_E = W/P_{\mathrm{loss}}$ and several reference
scalings at the same state, computed by the database's own definitions.

In [ ]:
references = {"ITER89P": "tau_e_ipb89_s", "IPB98(y,2)": "tau_e_h98y2_s", "NSTX H-mode": "tau_e_nstx_s",
              "NSTX 2006 L-mode": "tau_e_nstx2006l_s", "Kurskiev 2022": "tau_e_kurskiev2022_s"}
print(f"#{int(state['shot'])}: P_loss = {state['p_loss_MW'] * 1e3:.0f} kW, tau_E = {state['tau_e_s'] * 1e3:.2f} ms")
for name, column in references.items():
    print(f"  {name:17s} tau_E = {state[column] * 1e3:6.2f} ms   H = {state['tau_e_s'] / state[column]:.2f}")

alpha = {"I": 0.93, "B": 0.15, "P": -0.69, "n": 0.41, "R": 1.97}   # IPB98(y,2) engineering exponents
print(f"\nKadomtsev residual of the IPB98(y,2) exponents: "
      f"{eq_formula.kadomtsev_constraint_from_engineering_exponents(*alpha.values()):+.3f}")

One state gives five H factors spanning an order of magnitude, because the references were fitted
to different machines and regimes; only the ones whose domain includes a small ohmic spherical
tokamak are candidates, and even they are extrapolated. The H factor of a database is a
*distribution* over states, and it becomes a physics statement only when the $\tau_E$ definitions
(stored energy, loss power, $dW/dt$) are validated — the work of #548.

The Kadomtsev residual checks whether a set of engineering exponents is consistent with a
high-$\beta$ collisional similarity assumption: close to zero means consistent. It constrains which
fitted exponents make physical sense; it does not make a small sample sufficient to fit them.

### Step 10 — Correlation, regression and confounding

Fit $\ln l_i$ against $\ln q_{95}$ over the pooled slices, then the same fit within each shot.

In [ ]:
pooled = perform_ols_regression(slices.rename(columns={"li_3": "li"}), ["q_95"], target_param="li")
print(f"pooled over {len(slices)} slices: li ~ q95^{pooled.get_exponents()['q_95']:+.2f}")
for shot, group in slices.groupby("shot"):
    if len(group) < 4:
        print(f"  #{shot}: {len(group)} slice(s), no fit")
        continue
    within = perform_ols_regression(group.rename(columns={"li_3": "li"}), ["q_95"], target_param="li")
    print(f"  #{shot} alone ({len(group):2d} slices): li ~ q95^{within.get_exponents()['q_95']:+.2f}")
print(f"\ncorrelation of Ip and q95 over the pooled slices: {slices['ip_kA'].corr(slices['q_95']):+.2f}")

The pooled exponent is not an average over shots: it sits almost on 41672's own, because that
shot supplies half the slices — Step 1's pseudo-replication, now inside a fit. Within one
discharge, $q_{95}$ and $l_i$ both evolve as the current ramps and the plasma shrinks; between
discharges, they differ because the machine was run differently. A pooled fit mixes the two, and
the shot is a **confounder**. The $I_p$–$q_{95}$ correlation printed above is partly definitional: $q_{95}$
contains $I_p$. Before any regression on a database, decide whether rows are independent, whether
regressors share variables, and whether the fitted and tested states can come from the same shot
(train/test leakage).

### Step 11 — Similar states

Standardize each feature, $z_j = (x_j - \mu_j)/\sigma_j$, and measure the distance between states,
$d_{ij} = \lVert\mathbf z_i - \mathbf z_j\rVert$. Which other shot's state is nearest to each
representative state?

In [ ]:
features = ["q_95", "li_3", "elongation", "triangularity", "aspect_ratio"]
complete = slices.dropna(subset=features)   # a state missing a feature has no distance at all
standardized = (complete[features] - complete[features].mean()) / complete[features].std()
for shot, row_label in peak_rows.items():
    if row_label not in standardized.index:
        print(f"#{shot}: its max-Ip state lacks a feature -> not searched")
        continue
    here = standardized.loc[row_label]
    others = standardized[complete["shot"] != shot]
    distance = np.sqrt(((others - here) ** 2).sum(axis=1))
    nearest = distance.idxmin()
    print(f"#{shot} max-Ip state -> nearest other-shot state: #{int(complete.loc[nearest, 'shot'])} "
          f"at {complete.loc[nearest, 'time_s'] * 1e3:.1f} ms (d = {distance.min():.2f})")

The answer depends on choices the distance does not show: which features, how they are scaled,
that correlated features ($q_{95}$ and $l_i$ both follow the current) count twice, and that a
missing feature removes a state entirely. A distance also knows nothing about which dimension
matters for the physics at hand. Feature-space, distance and neighbour primitives with those
choices made explicit are #1114.

### Step 12 — Events and the state before them

Session 04 described a current quench as a sequence of numbers. Measure it on every shot that has a
current record, then join it to the **latest equilibrium slice before the quench began** — by time,
never by position.

In [ ]:
events = []
for shot, ods in samples.items():
    if "magnetics.ip.0.data" not in ods:
        print(f"#{shot}: no plasma-current record -> no event row")
        continue
    quench = current_quench(np.asarray(ods["magnetics.ip.0.time"], dtype=float),
                            np.asarray(ods["magnetics.ip.0.data"], dtype=float))
    # Step 2's rule already cut every slice from t_80 on; the latest remaining one is the pre-event state.
    before = slices[(slices["shot"] == shot) & (slices["time_s"] < quench.time_80)]
    state_row = before.loc[before["time_s"].idxmax()] if len(before) else None
    events.append({"shot": shot, "t_80_ms": quench.time_80 * 1e3, "t_80_20_ms": quench.duration_80_20 * 1e3,
                   "pre_state_ms": state_row["time_s"] * 1e3 if state_row is not None else np.nan,
                   "pre_q95": state_row["q_95"] if state_row is not None else np.nan,
                   "pre_li": state_row["li_3"] if state_row is not None else np.nan})
print(pd.DataFrame(events).round(2).to_string(index=False))

Each row joins an event-level record to a slice-level one, with the gap between them visible. The
join asks "what was the plasma like just before?", and the answer is only as good as the slice
cadence: a state milliseconds before a 10 ms quench is a pre-event state; a state long before it is
not. Distinguish **occupancy** — where states were — from the conditional probability
$P(\text{event} \mid \mathbf x)$, which needs event *and* non-event states from the same population.
A general event-record contract and a time-aware pre-event join are #1234.

**Stability labels** join the same way: each state could carry session 05's verdict — stable, ideally
unstable, tearing unstable, solver unavailable, rejected. Today only the legacy 39915 g-files have
DCON and RDCON runs (and only in lab mode); a population with stability labels is the aim of #940 and
#1084, and the reason session 05 exists.

## Interpretation Checkpoints

1. **More than half of the slices come from one shot.** What does that do to a histogram of $q_{95}$,
   and what would you plot instead?
2. **Two representative-state rules pick different slices on one shot.** Which rule would you use to
   compare against the $(q_a, l_i)$ survey, and why?
3. **No VEST state lies near $q = 2$.** Name three reasons that are not "it is unstable there", and
   the evidence that would separate them.
4. **The stored $\beta_N$ is a few hundredths.** Why is that a statement about the reconstruction
   rather than the plasma, and which session explained why?
5. **A function in `vaft.formula.stability` returns a "kink margin".** What did you check before
   drawing it, and what did you find?
6. **Five reference scalings give H factors spanning a factor of ten.** Which of them may be compared
   with VEST at all?
7. **The pooled $l_i$–$q_{95}$ exponent differs from the within-shot ones.** Which variable confounds
   the pooled fit?
8. **The nearest neighbour of a state changes when you drop one feature.** Is the search wrong?

## Integrated Analysis

### Part II, level 1 — the VEST operational-space study on the database

The previous VEST operating-characteristics analysis placed its discharges in $(q_a, l_i)$ and
$(q_a, \beta_N)$. Reproducing it is not redrawing its figures: it is the chain

```text
manually assembled research analysis -> canonical VAFT summaries -> reproducible population analysis
```

and the cells of Part I are that chain. In lab mode, swap the load line for the database summary
and rerun: coverage, validity, the representative rule, the $(q, l_i)$ plane and the regression all
follow. Compare three populations — the one the publication used, the current database, and what
changed in reconstruction coverage between them.

In [ ]:
LAB = os.environ.get("VAFT_TUTORIAL_MODE", "offline") == "lab"
if LAB:
    # The summary skips a shot it cannot open and says why in the log (a 403 is missing credentials);
    # keep those messages visible here, or an unreachable database reads as an empty one.
    logging.getLogger("vaft.database").setLevel(logging.WARNING)
    lab_slices = vaft.database.summary(shot_range=(39000, 49000), preset="equilibrium_global")
    if lab_slices.empty:
        raise RuntimeError("the database returned no equilibrium rows: check the logged reasons above")
    lab_slices = lab_slices[lab_slices["ip_kA"].abs() > 0]
    print(f"database population: {len(lab_slices)} slices from {lab_slices['shot'].nunique()} shots")
    figure, axes = plt.subplots(figsize=(7.5, 5))
    axes.plot(lab_slices["q_95"], lab_slices["li_3"], ".", alpha=0.3, label="database slices")
    axes.plot(qa_survey, li_survey, "x", color="0.55", label="JET survey points, reference only")
    axes.set_xlabel(r"$q_{95}$")
    axes.set_ylabel(r"$l_i$")
    axes.legend()
    plt.show()
else:
    print("Offline: the database population needs lab mode (VAFT_TUTORIAL_MODE=lab and HSDS credentials).\n"
          "Every step of Part I reruns on it unchanged.")

### Part II, level 2 — from statistics to learned representations (planned)

Everything above uses **hand-chosen features**: $q_{95}$, $l_i$, $\kappa$. The data-driven question
is whether the relevant representation must always be chosen by hand:

$$
\boxed{\text{descriptive statistics}} \rightarrow \boxed{\text{machine learning}}
\rightarrow \boxed{\text{representation learning}}
$$

**Why scalars are not enough.** Two discharges with the same maximum $I_p$ can differ in ramp rate,
oscillations, mode onset, chirping and termination. A representative feature vector loses exactly
that information.

| level | representation | model families |
| --- | --- | --- |
| 1 | scalar / tabular state $[I_p, q, \beta, l_i, \dots]$ | linear, logistic, trees, shallow MLP |
| 2 | diagnostic time series $X \in \mathbb R^{T\times C}$ | 1-D CNN, TCN, recurrent, Transformer |
| 3 | profiles $X(\rho, t)$ | profile encoders, sequence models, neural operators |
| 4 | images $I(R, Z, t)$ | CNN, video encoders |
| 5 | multimodal state → $\mathbf z_{\mathrm{plasma}}$ | multimodal encoders |

**Planned — not implemented as a validated VAFT workflow:**

| task | input | output |
| --- | --- | --- |
| discharge classification | features or signals | regime label |
| IRE / disruption prediction | pre-event history | event risk |
| time-to-event | state history | hazard $h(t \mid \mathbf x)$ |
| stability surrogate | equilibrium, profiles | $\delta W$, $\Delta'$, class |
| similar-shot retrieval | state embedding | nearest cases |
| anomaly detection | state, diagnostics | anomaly score |
| operational-boundary learning | database | risk surface $P(\text{event within }\Delta t \mid \mathbf x)$ |

`vaft.process.ml` provides the infrastructure such work needs — fingerprinted datasets, group-aware
splits so that no shot lands on both sides, reproducible training records — and the IRE and SXR
contracts are its first users. It does not provide production plasma models, and this session does
not pretend otherwise. The rule already met in Step 10 carries over unchanged: **split by shot**, or
the test set leaks.

The conceptual endpoint is a *latent operational space*: a learned map $\mathbf x \rightarrow
\mathbf z \in \mathbb R^2$ annotated with stable states, instabilities, IREs, terminations and good
confinement — the high-dimensional successor of the $(q_a, l_i)$ plane.

## Independent Exercise

### A population study of your own question

Choose a question that needs many states — for example "does VEST approach low $q$ at high $l_i$?",
"how does $\kappa$ vary with $I_p$?", or "which discharges resemble 41672 at its current peak?" —
and, in lab mode on a database shot range (or offline on the six samples, saying so):

1. State the **unit of analysis** and the **sampling rule** for your representative state.
2. Build the tables from the summary presets and report the **coverage funnel** and every validity
   rule you apply, with counts.
3. Report the **distribution** of each quantity you use, per state and per shot.
4. Draw the **operational space** your question needs, and label every boundary as observed or
   referenced; give each reference's source and domain.
5. If you use $\beta$, $\tau_E$ or density, state the **validity** of that column for your population.
6. Fit what you need, **within and across shots**, and name the confounders.
7. If you use events, join them **by time** and give the gap to the pre-event state.
8. Say what a **stability label** or a **learned representation** would add, and what VAFT would
   need to provide it.

In [ ]:
# Lab mode: VAFT_TUTORIAL_MODE=lab and HSDS credentials.
# tables = {preset: vaft.database.summary(shot_range=(<first>, <last>), preset=preset) for preset in PRESETS}
# slices = tables["equilibrium_global"]
#
# A representative rule of your own, e.g. the slice nearest a fixed fraction of the pulse:
# chosen = slices.loc[slices.groupby("shot")["<column>"].idxmax()]
#
# Within-shot and pooled fits:
# perform_ols_regression(slices.rename(columns={"li_3": "li"}), ["q_95", "elongation"], target_param="li")
#
# Other presets: "neoclassical", "efit_reliability" (per-channel EFIT quality) -- check coverage first.

## Takeaways and Next Steps

- A database study starts when one quantity has one definition and one provenance in every row.
  VAFT's summary presets are that definition; the tutorial adds none of its own.
- The unit of analysis is a choice — shot, slice, event — and so is the representative state.
  Slices of one shot are not independent samples of the machine.
- Coverage and validity come first. Missing rows, failed presets, zero-current slices and a
  pressure-free reconstruction all change what a population can say.
- **An observed envelope is not an operating limit.** A reference limit is drawn only with its source
  and domain; a formula existing is not evidence (#350).
- Engineering variables become dimensionless ones before machines — or regimes — are compared.
  One $\tau_E$ against five scalings is five questions, not one answer (#548).
- Pooled fits across shots are confounded by the shot; split by shot for tests and for learning.
- Statistics, machine learning and representation learning are one ladder. VAFT has the first rung,
  the infrastructure for the second, and a plan for the third.

**Open work this session depends on**: representative-state and coverage APIs (#1233), in-memory
summaries (#1232), event contracts and pre-event joins (#1234), population operational-space plots
(#944), confinement from validated power balance (#548), density-limit families (#1068), feature-space
and neighbour primitives (#1114), stability labels in aggregate (#940, #1084).

**This is the end of the course.** Session 01 opened one discharge; session 06 has put it among the
others. The next step is your own question on the database.